# Notebook 03j — Optimización de hiperparámetros con Optuna (target: TUMOR)

**Objetivo**: encontrar los hiperparámetros óptimos para Lasso, Ridge y Elastic Net usando Optuna,  
con las mismas features que 03c (verdes + azules + mesenquimal, target: fusiones en TUMOR).  
**Referencia**: 03c con alpha fijo → Lasso R²_CV=0.0554, Ridge R²_CV=0.0347, ElasticNet R²_CV=0.0455.  
**Método**: Optuna busca el alpha óptimo en un rango continuo usando 5-fold CV como función objetivo.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # silenciar logs de Optuna

from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score

import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(f'CWD: {os.getcwd()}')
print('Librerías cargadas OK')

## 2. Carga de datos (idéntica a 03c)

In [ ]:
TARGET = 'Tumor_Telomere fusion rate (Gt-end/coverage*purity)'

df = pd.read_csv('datos_limpios_modelo_a.csv')

df_verde = pd.read_excel('revision_color_code.xlsx', sheet_name='Sheet1',
                         usecols=['Patient_ID_IF_mod', 'firstResponse', 'treatment'])
df = df.merge(df_verde, on='Patient_ID_IF_mod', how='left')

df_azul = pd.read_excel('revision_color_code.xlsx', sheet_name='Sheet1',
                        usecols=['Patient_ID_IF_mod', 'biopsySite', 'hasSystemicPreTreatment',
                                 'hasRadiotherapyPreTreatment', 'treatmentGiven', 'treatment1'])
df = df.merge(df_azul, on='Patient_ID_IF_mod', how='left')

df_mes = pd.read_excel('Tumor_mesenchimal.xlsx')
df_mes.columns = ['cancer_type_code', 'Mesenchimal_origin']
df = df.merge(df_mes, on='cancer_type_code', how='left')

print(f'Shape final: {df.shape}')

## 3. Target y features (idénticos a 03c)

In [ ]:
y = np.log1p(df[TARGET])

df['firstResponse'] = df['firstResponse'].fillna('Sin_dato')
df['Mesenchimal_origin'] = df['Mesenchimal_origin'].fillna('Sin_dato')
top15 = df['treatment'].value_counts().head(15).index.tolist()
df['treatment_agrupado'] = df['treatment'].apply(
    lambda x: x if x in top15 else ('Sin_dato' if pd.isna(x) else 'Otro')
)
df['hasSystemicPreTreatment'] = df['hasSystemicPreTreatment'].fillna('Sin_dato')
df['hasRadiotherapyPreTreatment'] = df['hasRadiotherapyPreTreatment'].fillna('Sin_dato')
df['treatmentGiven'] = df['treatmentGiven'].fillna('Sin_dato')
top10_biopsy = df['biopsySite'].value_counts().head(10).index.tolist()
df['biopsySite_agrupado'] = df['biopsySite'].apply(
    lambda x: x if x in top10_biopsy else ('Sin_dato' if pd.isna(x) else 'Otro')
)
top10_t1 = df['treatment1'].value_counts().head(10).index.tolist()
df['treatment1_agrupado'] = df['treatment1'].apply(
    lambda x: x if x in top10_t1 else ('Sin_dato' if pd.isna(x) else 'Otro')
)

cols_features = ['firstResponse', 'treatment_agrupado', 'Mesenchimal_origin',
                 'cancer_type', 'biopsySite_agrupado', 'hasSystemicPreTreatment',
                 'hasRadiotherapyPreTreatment', 'treatmentGiven', 'treatment1_agrupado']

X = pd.get_dummies(df[cols_features], drop_first=True)
print(f'Shape X: {X.shape} | Shape y: {y.shape}')

## 4. Configuración CV

Usamos el mismo KFold que en 03c para que los resultados sean comparables directamente.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
N_TRIALS = 100  # número de combinaciones que prueba Optuna
print(f'KFold: 5 folds | Trials Optuna: {N_TRIALS}')

## 5. Optuna — Lasso

Optuna busca el alpha óptimo en el rango [1e-4, 10] en escala logarítmica.
La función objetivo es el R² medio de 5-fold CV (queremos maximizarlo).

In [ ]:
def objective_lasso(trial):
    alpha = trial.suggest_float('alpha', 1e-4, 10.0, log=True)
    pipe  = Pipeline([('scaler', StandardScaler()),
                      ('model',  Lasso(alpha=alpha, max_iter=10000, random_state=42))])
    scores = cross_val_score(pipe, X, y, cv=kf, scoring='r2')
    return scores.mean()

study_lasso = optuna.create_study(direction='maximize')
study_lasso.optimize(objective_lasso, n_trials=N_TRIALS, show_progress_bar=False)

best_alpha_lasso = study_lasso.best_params['alpha']
best_r2_lasso    = study_lasso.best_value
print(f'Lasso — mejor alpha: {best_alpha_lasso:.6f}  |  R² CV: {best_r2_lasso:.4f}')
print(f'Referencia 03c (alpha=0.001):  R² CV = 0.0554')

## 6. Optuna — Ridge

In [ ]:
def objective_ridge(trial):
    alpha = trial.suggest_float('alpha', 1e-2, 1000.0, log=True)
    pipe  = Pipeline([('scaler', StandardScaler()),
                      ('model',  Ridge(alpha=alpha, random_state=42))])
    scores = cross_val_score(pipe, X, y, cv=kf, scoring='r2')
    return scores.mean()

study_ridge = optuna.create_study(direction='maximize')
study_ridge.optimize(objective_ridge, n_trials=N_TRIALS, show_progress_bar=False)

best_alpha_ridge = study_ridge.best_params['alpha']
best_r2_ridge    = study_ridge.best_value
print(f'Ridge — mejor alpha: {best_alpha_ridge:.4f}  |  R² CV: {best_r2_ridge:.4f}')
print(f'Referencia 03c (alpha=100.0):  R² CV = 0.0347')

## 7. Optuna — Elastic Net

In [ ]:
def objective_enet(trial):
    alpha    = trial.suggest_float('alpha',    1e-4, 10.0, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.0,  1.0)
    pipe = Pipeline([('scaler', StandardScaler()),
                     ('model',  ElasticNet(alpha=alpha, l1_ratio=l1_ratio,
                                           max_iter=10000, random_state=42))])
    scores = cross_val_score(pipe, X, y, cv=kf, scoring='r2')
    return scores.mean()

study_enet = optuna.create_study(direction='maximize')
study_enet.optimize(objective_enet, n_trials=N_TRIALS, show_progress_bar=False)

best_alpha_enet    = study_enet.best_params['alpha']
best_l1_enet       = study_enet.best_params['l1_ratio']
best_r2_enet       = study_enet.best_value
print(f'Elastic Net — mejor alpha: {best_alpha_enet:.6f}  |  l1_ratio: {best_l1_enet:.4f}  |  R² CV: {best_r2_enet:.4f}')
print(f'Referencia 03c (alpha=0.001, l1=0.5):  R² CV = 0.0455')

## 8. Comparativa final

In [ ]:
sep = '-'
print('=== COMPARATIVA — alpha fijo (03c) vs Optuna (03j) ===')
print(f'  {"Modelo":<38}  {"Alpha":<12}  {"R2 CV":>8}')
print(f'  {sep*38}  {sep*12}  {sep*8}')
print(f'  {"Lasso       (03c, alpha=0.001)":<38}  {"0.001":<12}  {0.0554:>8.4f}')
print(f'  {"Lasso       (Optuna)":<38}  {best_alpha_lasso:<12.6f}  {best_r2_lasso:>8.4f}')
print(f'  {"Ridge       (03c, alpha=100.0)":<38}  {"100.0":<12}  {0.0347:>8.4f}')
print(f'  {"Ridge       (Optuna)":<38}  {best_alpha_ridge:<12.4f}  {best_r2_ridge:>8.4f}')
print(f'  {"Elastic Net (03c, a=0.001, l1=0.5)":<38}  {"0.001/0.5":<12}  {0.0455:>8.4f}')
print(f'  {"Elastic Net (Optuna)":<38}  {str(round(best_alpha_enet,4))+"/"+str(round(best_l1_enet,2)):<12}  {best_r2_enet:>8.4f}')
print()
mejora_lasso = best_r2_lasso - 0.0554
print(f'Mejora Lasso vs baseline: {mejora_lasso:+.4f}')
ganador = max({'Lasso': best_r2_lasso, 'Ridge': best_r2_ridge, 'ElasticNet': best_r2_enet},
              key=lambda k: {'Lasso': best_r2_lasso, 'Ridge': best_r2_ridge, 'ElasticNet': best_r2_enet}[k])
r2_ganador = max(best_r2_lasso, best_r2_ridge, best_r2_enet)
print(f'Mejor modelo optimizado: {ganador}  (R² CV = {r2_ganador:.4f})')

## 9. Curva de optimización Lasso

Visualiza cómo evoluciona el mejor R² a medida que Optuna prueba más valores de alpha.
Si la curva se estabiliza, significa que 100 trials son suficientes.

In [ ]:
trials_lasso = study_lasso.trials
valores      = [t.value for t in trials_lasso]
mejor_acum   = pd.Series(valores).cummax()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Curva de convergencia
axes[0].plot(mejor_acum.values, color='steelblue', linewidth=1.5)
axes[0].axhline(0.0554, color='red', linestyle='--', linewidth=1, label='Baseline 03c')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Mejor R² CV acumulado')
axes[0].set_title('Convergencia Optuna — Lasso')
axes[0].legend()

# Alpha vs R² para cada trial
alphas_tried = [t.params['alpha'] for t in trials_lasso]
r2_tried     = [t.value for t in trials_lasso]
axes[1].scatter(alphas_tried, r2_tried, alpha=0.4, s=15, color='steelblue')
axes[1].axvline(best_alpha_lasso, color='red', linestyle='--', linewidth=1,
                label=f'Mejor alpha={best_alpha_lasso:.5f}')
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha (escala log)')
axes[1].set_ylabel('R² CV')
axes[1].set_title('Alpha vs R² — Lasso')
axes[1].legend()

plt.suptitle('Optimización Optuna — Lasso (target: TUMOR)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('notebooks/img/img_03j_optuna_lasso.png', dpi=150, bbox_inches='tight')
plt.close()
print('Guardado: notebooks/img/img_03j_optuna_lasso.png')

---

# PARTE 2 — Optuna con target: SANGRE (referencia: 03h)

Las features son exactamente las mismas que arriba (tratamientos verdes + azules + mesenquimal).
Solo cambia el target: ahora predecimos las fusiones telomericas en **sangre periferica**.

**Referencia 03h con alpha fijo**: Lasso R²_CV=0.0531, Ridge R²_CV=0.0347, ElasticNet R²_CV=0.0455  
**Es el modelo mas estable de todos**: std=0.0134 (vs 0.0282 del tumor)

## 10. Target sangre — log1p

In [ ]:
TARGET_SANGRE = 'Blood_Telomere fusion rate (Gt-end/coverage*purity)'

df_final = pd.read_csv('datos_modelo_final.csv',
                       usecols=['Patient_ID_IF_mod', TARGET_SANGRE])

# Alineamos con el df actual (mismo orden de pacientes que X)
df_s = df[['Patient_ID_IF_mod']].merge(df_final, on='Patient_ID_IF_mod', how='inner')
y_s  = np.log1p(df_s[TARGET_SANGRE])

# X_s: mismas features, solo los pacientes con target de sangre disponible
idx_s = df_s.index
X_s   = X.loc[idx_s].reset_index(drop=True)
y_s   = y_s.reset_index(drop=True)

print(f'Pacientes con target sangre: {len(y_s)}')
print(f'Shape X_s: {X_s.shape} | Shape y_s: {y_s.shape}')
print(f'Skewness log1p sangre: {y_s.skew():.2f}')

## 11. Optuna — Lasso (sangre)

In [ ]:
def objective_lasso_s(trial):
    alpha = trial.suggest_float('alpha', 1e-4, 10.0, log=True)
    pipe  = Pipeline([('scaler', StandardScaler()),
                      ('model',  Lasso(alpha=alpha, max_iter=10000, random_state=42))])
    scores = cross_val_score(pipe, X_s, y_s, cv=kf, scoring='r2')
    return scores.mean()

study_lasso_s = optuna.create_study(direction='maximize')
study_lasso_s.optimize(objective_lasso_s, n_trials=N_TRIALS, show_progress_bar=False)

best_alpha_lasso_s = study_lasso_s.best_params['alpha']
best_r2_lasso_s    = study_lasso_s.best_value
print(f'Lasso (sangre) — mejor alpha: {best_alpha_lasso_s:.6f}  |  R² CV: {best_r2_lasso_s:.4f}')
print(f'Referencia 03h (alpha=0.001):  R² CV = 0.0531')

## 12. Optuna — Ridge (sangre)

In [ ]:
def objective_ridge_s(trial):
    alpha = trial.suggest_float('alpha', 1e-2, 1000.0, log=True)
    pipe  = Pipeline([('scaler', StandardScaler()),
                      ('model',  Ridge(alpha=alpha, random_state=42))])
    scores = cross_val_score(pipe, X_s, y_s, cv=kf, scoring='r2')
    return scores.mean()

study_ridge_s = optuna.create_study(direction='maximize')
study_ridge_s.optimize(objective_ridge_s, n_trials=N_TRIALS, show_progress_bar=False)

best_alpha_ridge_s = study_ridge_s.best_params['alpha']
best_r2_ridge_s    = study_ridge_s.best_value
print(f'Ridge (sangre) — mejor alpha: {best_alpha_ridge_s:.4f}  |  R² CV: {best_r2_ridge_s:.4f}')
print(f'Referencia 03h (alpha=1.0):  R² CV = 0.0347')

## 13. Optuna — Elastic Net (sangre)

In [ ]:
def objective_enet_s(trial):
    alpha    = trial.suggest_float('alpha',    1e-4, 10.0, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.0,  1.0)
    pipe = Pipeline([('scaler', StandardScaler()),
                     ('model',  ElasticNet(alpha=alpha, l1_ratio=l1_ratio,
                                           max_iter=10000, random_state=42))])
    scores = cross_val_score(pipe, X_s, y_s, cv=kf, scoring='r2')
    return scores.mean()

study_enet_s = optuna.create_study(direction='maximize')
study_enet_s.optimize(objective_enet_s, n_trials=N_TRIALS, show_progress_bar=False)

best_alpha_enet_s = study_enet_s.best_params['alpha']
best_l1_enet_s    = study_enet_s.best_params['l1_ratio']
best_r2_enet_s    = study_enet_s.best_value
print(f'Elastic Net (sangre) — mejor alpha: {best_alpha_enet_s:.6f}  |  l1_ratio: {best_l1_enet_s:.4f}  |  R² CV: {best_r2_enet_s:.4f}')
print(f'Referencia 03h (alpha=0.001, l1=0.5):  R² CV = 0.0455')

## 14. Comparativa final — SANGRE vs baseline 03h

In [ ]:
sep = '-'
print('=== COMPARATIVA — alpha fijo (03h) vs Optuna — TARGET: SANGRE ===')
print(f'  {"Modelo":<38}  {"Alpha":<12}  {"R2 CV":>8}')
print(f'  {sep*38}  {sep*12}  {sep*8}')
print(f'  {"Lasso       (03h, alpha=0.001)":<38}  {"0.001":<12}  {0.0531:>8.4f}')
print(f'  {"Lasso       (Optuna)":<38}  {best_alpha_lasso_s:<12.6f}  {best_r2_lasso_s:>8.4f}')
print(f'  {"Ridge       (03h, alpha=1.0)":<38}  {"1.0":<12}  {0.0347:>8.4f}')
print(f'  {"Ridge       (Optuna)":<38}  {best_alpha_ridge_s:<12.4f}  {best_r2_ridge_s:>8.4f}')
print(f'  {"Elastic Net (03h, a=0.001, l1=0.5)":<38}  {"0.001/0.5":<12}  {0.0455:>8.4f}')
print(f'  {"Elastic Net (Optuna)":<38}  {str(round(best_alpha_enet_s,4))+"/"+str(round(best_l1_enet_s,2)):<12}  {best_r2_enet_s:>8.4f}')
print()
r2_ganador_s = max(best_r2_lasso_s, best_r2_ridge_s, best_r2_enet_s)
print(f'Mejor modelo optimizado (sangre): R² CV = {r2_ganador_s:.4f}')
print()
print('--- Resumen global Optuna ---')
print(f'  TUMOR  — mejor R² CV: {max(best_r2_lasso, best_r2_ridge, best_r2_enet):.4f}  (baseline 03c: 0.0554)')
print(f'  SANGRE — mejor R² CV: {r2_ganador_s:.4f}  (baseline 03h: 0.0531)')